# 04. Was the rush bigger where the tax break was worth more?

**COMM 1280 Final Data Project** · Jesus Aguirre and Lisandro Collazo

> **Status: started, not finished.** The code below runs. The analysis after it
> is the work still to do. This notebook is separate so two people can work at
> the same time without editing the same file.

## The idea

The tax break forgave tax on the **value of the building**, so it was worth
more on an expensive house than a cheap one. Someone building a $600,000 home
had far more to lose by missing the deadline than someone building a $200,000
home.

If the December 2021 rush really was about the tax break, it should have been
**strongest in the most expensive parts of the city**. If the rush looks the
same everywhere, something else was going on.

This is the best test left of our story.

In [ ]:
import pandas as pd
import requests

API = 'https://phl.carto.com/api/v2/sql'
RAW = '../data/raw/'

permits = pd.read_csv(RAW + 'new_construction_building_permits_2016_2024.csv')
permits['month'] = permits['permitissuedate'].str[:7]

homes = permits[permits['permitdescription'] == 'Residential Building Permit']

print('home permits:', len(homes))
print('that have a property account number:', homes['opa_account_num'].notna().sum())
print('that have a census tract:', homes['censustract'].notna().sum())

## Step 1. Get the property values

The Office of Property Assessment publishes what the city thinks every
property is worth. The number we want is the value of the building itself,
because that is what the tax break applied to.

The table is called `opa_properties_public`. Our permits connect to it through
the `opa_account_num` column.

Run the cell below to see which columns are available, then pick the value
column to use.

In [ ]:
sql = "SELECT * FROM opa_properties_public LIMIT 1"
response = requests.get(API, params={'q': sql})
columns = sorted(response.json()['rows'][0].keys())

print('total columns:', len(columns))
print()
for name in columns:
    if 'value' in name or 'market' in name or 'exempt' in name:
        print(name)

### To do

1. Pick the market value column from the list above.
2. Download it for Philadelphia's residential properties.
3. Save it as `../data/raw/property_assessments.csv`.

A starting point, to finish once the column names are chosen:

In [ ]:
# sql = """
# SELECT parcel_number, market_value, census_tract, year_built
# FROM opa_properties_public
# WHERE category_code_description LIKE '%Residential%'
# """
#
# response = requests.get(API, params={'q': sql, 'format': 'csv'})
# outfile = open(RAW + 'property_assessments.csv', 'w', encoding='utf-8')
# outfile.write(response.text)
# outfile.close()

## Step 2. Sort neighbourhoods by how expensive they are

Join the property values onto the permits, matching `opa_account_num` to
`parcel_number`. Then sort the census tracts into four groups, from the
cheapest quarter of the city to the most expensive quarter.

## Step 3. Compare the rush in each group

For each of the four groups, work out the same number we used in notebook 03:

```
permits in December 2021  divided by  the average month, January to October 2021
```

**If our story is right:** that number gets bigger as the neighbourhoods get
more expensive.

**If our story is wrong:** the number is about the same everywhere.

## Step 4. Add the Census data

Bring in median household income by census tract from the American Community
Survey, and check whether the pattern follows income too. This is the third
dataset the assignment asks for.